<a href="https://colab.research.google.com/github/88AJ/liturgiapro.com/blob/main/Backend_Server_(Flask_Jinja2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import subprocess
from flask import Flask, request, send_file, jsonify
import jinja2

app = Flask(__name__)

# 1. Configure Jinja2 to avoid conflicts with LaTeX curly braces {}
latex_jinja_env = jinja2.Environment(
    block_start_string='\BLOCK{',
    block_end_string='}',
    variable_start_string='\VAR{',
    variable_end_string='}',
    comment_start_string='\#{',
    comment_end_string='}',
    trim_blocks=True,
    autoescape=False,
    loader=jinja2.FileSystemLoader(os.path.abspath('.'))
)

# 2. Data Sanitization Functions
def escape_latex(text):
    if not text:
        return ""

    # Escape special LaTeX characters
    escape_map = {
        '&': r'\&', '%': r'\%', '$': r'\$', '#': r'\#', '_': r'\_',
        '{': r'\{', '}': r'\}', '~': r'\textasciitilde{}', '^': r'\textasciicircum{}', '\\': r'\textbackslash{}'
    }

    # Process text character by character
    escaped_text = "".join(escape_map.get(c, c) for c in text)

    # Convert standard newlines to LaTeX line breaks (for sense lines)
    escaped_text = escaped_text.replace('\n', r' \\' + '\n')
    return escaped_text

def apply_lettrine(text):
    if not text:
        return ""
    text = text.strip()
    match = re.match(r'^(¿?¡?[A-Za-záéíóúÁÉÍÓÚñÑ])(.*)', text)
    if match:
        first_char = match.group(1)
        rest_of_word = match.group(2).split(' ', 1)

        # Format: \lettrine{F}{irst} rest of the sentence
        if len(rest_of_word) > 1:
            return f"\\lettrine{{{first_char}}}{{{rest_of_word[0]}}} {rest_of_word[1]}"
        else:
            return f"\\lettrine{{{first_char}}}{{{rest_of_word[0]}}}"
    return text

# Register filters in Jinja environment
latex_jinja_env.filters['escape_tex'] = escape_latex
latex_jinja_env.filters['lettrine'] = apply_lettrine

@app.route('/generate-pdf', methods=['POST'])
def generate_pdf():
    data = request.json

    if not data:
        return jsonify({"error": "No JSON payload provided"}), 400

    template = latex_jinja_env.get_template('template.tex')

    # Render .tex file
    rendered_tex = template.render(**data)

    tex_filename = "temp_subsidio.tex"
    pdf_filename = "temp_subsidio.pdf"

    with open(tex_filename, "w", encoding="utf-8") as f:
        f.write(rendered_tex)

    # 3. Compile PDF using XeLaTeX
    try:
        subprocess.run(
            ["xelatex", "-interaction=nonstopmode", tex_filename],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE
        )
    except subprocess.CalledProcessError as e:
        return jsonify({"error": "LaTeX compilation failed", "logs": e.stdout.decode()}), 500

    # 4. Return generated PDF
    if os.path.exists(pdf_filename):
        return send_file(pdf_filename, as_attachment=False, mimetype='application/pdf')
    else:
        return jsonify({"error": "PDF file not found after compilation"}), 500

if __name__ == '__main__':
    app.run(port=8086, debug=True)